# 01_parse: Nexis RTF to corpus.jsonl

> **Environment:** requires the project venv **`.venv311`** (Python 3.11) as the Jupyter kernel — the pipeline dependencies are installed only there. `run_all.command` uses it automatically. See `README.md` → Environment setup.

**Input:** All `.rtf` files in `data/raw/`
**Output:** `data/interim/corpus.jsonl`
Never modifies `data/raw/`.

## Pipeline steps in this notebook

1. Setup & paths
2. RTF stripping (read `.rtf` to clean plain text)
3. Parser helpers (date, body, headline, source)
4. Parse all files in `data/raw/`
5. Quality report
6. Write on `corpus.jsonl`

## Nexis RTF article structure (after RTF stripping)

```
[ PREAMBLE ]                    <- User name, search query (block 0 only)
Documents (N)
[ NUMBERED TABLE OF CONTENTS ]  <- block 0 only

HEADLINE                        <- bare first line, no label
SOURCE NAME                     <- second line
MONTH D, YYYY Weekday [Edition] <- third line

Copyright YYYY Publisher
Section: ...
Length: N words
Byline: ...
[Highlight: ...]                <- optional summary line
Body

[ ARTICLE BODY TEXT ]

Classification                  <- Nexis metadata footer (NOT body content)
Language: ENGLISH
Subject: ...
Person: ...
Geographic: ...

Load-Date: ...

End of DocumentHEADLINE         <- delimiter then next article (no blank line)
...
```

**Critical:** the body field STOPS at `Classification` (or `Load-Date:` if no Classification).
The Classification block contains entity names like 'DONALD TRUMP', 'IRAN', 'TEHRAN' as Nexis taxonomy tags.
If left in the body, NER would harvest them as if they were narrative actors and concept matches would fire on metadata, badly inflating co-occurrence counts.

## Step 1: Setup & paths

In [ ]:
import re
import json
import html
import sys
from pathlib import Path

# Iterate starting from CWD until we find src/
_cwd = Path().resolve()
ROOT = next(
    (p for p in [_cwd] + list(_cwd.parents) if (p / 'src').is_dir()),
    _cwd,
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RAW_DIR     = ROOT / 'data' / 'raw'
INTERIM_DIR = ROOT / 'data' / 'interim'
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
OUT_FILE = INTERIM_DIR / 'corpus.jsonl'

print(f'Python       : {sys.executable}')
print(f'CWD          : {_cwd}')
print(f'Project root : {ROOT}')
print(f'Raw dir      : {RAW_DIR}')
print(f'Output file  : {OUT_FILE}')
assert (ROOT / 'src').is_dir(), f'ERROR: src/ not found under {ROOT}'

## Step 2: RTF stripping

Strip the RTF markup with `striprtf`, then unescape any embedded HTML entities

Needed: `pip install striprtf`

In [ ]:
try:
    from striprtf.striprtf import rtf_to_text
    HAS_STRIPRTF = True
    print('striprtf available')
except ImportError:
    HAS_STRIPRTF = False
    print('striprtf not installed')


def read_as_plaintext(path: Path) -> str:
    # Nexis only exports .rtf; refuse anything else to prevent silent mis-parses
    # if a stray .txt or .docx ends up in data/raw/.
    if path.suffix.lower() != '.rtf':
        raise ValueError(f'Only .rtf is supported by this pipeline. Got: {path.name}')
    if not HAS_STRIPRTF:
        raise ImportError(
            f'Cannot read {path.name}: striprtf required for .rtf files.\n'
            'Install with: pip install striprtf'
        )
    raw  = path.read_bytes().decode('latin-1', errors='replace')
    text = rtf_to_text(raw)
    return html.unescape(text)

## Step 3: Parser helpers

Three field extractors:
- **Date** via month-name regex (ignores weekday/edition suffix)
- **Body** stops at `Classification`, then `Load-Date:`, then end-of-block — whichever comes first
- **Headline + source** anchored on the date line, stepping back two rows

In [ ]:
MONTH_NAMES = (
    'January|February|March|April|May|June|'
    'July|August|September|October|November|December'
)
DATE_RE = re.compile(rf'((?:{MONTH_NAMES})\s+\d{{1,2}},\s+202\d)')

# Body: 'Body' on its own line, then blank lines, then article text.
# Stops at the first of: Classification, Graphic, Load-Date, end of block.
# This excludes the Nexis metadata footer from the body field.
BODY_RE = re.compile(
    r'\nBody\s*\n+(.*?)(?=\n(?:Classification|Graphic|Load-Date)|\Z)',
    re.DOTALL,
)

COPYRIGHT_RE = re.compile(r'\nCopyright 20\d\d')


def extract_date(text: str) -> str | None:
    m = DATE_RE.search(text)
    return m.group(1) if m else None


def extract_header_fields(header: str, date: str | None) -> tuple[str | None, str | None]:
    # Lines before the Copyright divider are: HEADLINE / SOURCE / DATE [edition]
    m = COPYRIGHT_RE.search(header)
    pre_copyright = header[: m.start()] if m else header
    lines = [l.strip() for l in pre_copyright.split('\n') if l.strip()]
    if not lines:
        return None, None

    # Anchor on the date line (unique within a single article block) and step back
    if date:
        date_key = date[:10]
        date_idx = next((i for i, l in enumerate(lines) if date_key in l), None)
        if date_idx is not None:
            source   = lines[date_idx - 1] if date_idx >= 1 else None
            headline = lines[date_idx - 2] if date_idx >= 2 else None
            return headline, source

    # Fallback when no date is found
    headline = lines[-2] if len(lines) >= 2 else lines[0]
    source   = lines[-1] if len(lines) >= 2 else None
    return headline, source


def parse_block(block: str, article_id: int) -> dict | None:
    block = block.strip()
    if not block:
        return None

    body_match = BODY_RE.search(block)
    if not body_match:
        return None

    body = body_match.group(1).strip()
    if len(body.split()) < 10:
        return None

    header            = block[: body_match.start()]
    date              = extract_date(header)
    headline, source  = extract_header_fields(header, date)

    return {
        'id':       f'{article_id:04d}',
        'date':     date,
        'source':   source,
        'headline': headline,
        'body':     body,
    }

## Step 4: Parse all files in data/raw/

Articles are split on `End of Document`. Each article block is parsed exactly once. 

In [ ]:
# Nexis exports .rtf only; scan exclusively for that suffix.
input_files = sorted({
    p for p in RAW_DIR.iterdir()
    if p.is_file() and p.suffix.lower() == '.rtf'
})
print(f'Found {len(input_files)} .rtf file(s) in {RAW_DIR}')
for f in input_files:
    print(f'  {f.name}  ({f.stat().st_size // 1024} KB)')

if not input_files:
    print('WARNING: No .rtf files found. Place your Nexis exports in data/raw/ and re-run.')

In [ ]:
all_records = []
global_id   = 1

for file_path in input_files:
    print(f'\nProcessing: {file_path.name}')

    try:
        plain = read_as_plaintext(file_path)
    except Exception as e:
        print(f'  ERROR reading file: {e}')
        continue

    # Each block between two End of Document markers is one article.
    # blocks[0] also includes the Nexis preamble + TOC; that prefix gets discarded because BODY_RE only matches the article body section.
    blocks = plain.split('End of Document')
    print(f'  Blocks after split  : {len(blocks)}')

    file_records = []
    for block in blocks:
        record = parse_block(block, global_id)
        if record is not None:
            file_records.append(record)
            global_id += 1

    print(f'  Articles parsed     : {len(file_records)}')
    all_records.extend(file_records)

print(f'\nTotal records parsed across all files: {len(all_records)}')

## Step 5: Quality report

Check whether bodies still contain `Classification`, `Subject:`, `Geographic:` (or step 3 failed).

In [ ]:
null_date     = [r for r in all_records if r['date']     is None]
null_source   = [r for r in all_records if r['source']   is None]
null_headline = [r for r in all_records if r['headline'] is None]
# Nexis taxonomy lines carry "(NN%)" relevance scores — that is the metadata
# signature. A bare "\nSubject:" also occurs in genuine article content (e.g.
# quoted diplomatic letters in IRNA copy), so only the taxonomy form counts.
NEXIS_TAXONOMY_RE = re.compile(r'\n(?:Subject|Geographic):[^\n]*\(\d{1,3}%\)')
leaked_meta   = [r for r in all_records
                 if r['body'] and ('\nClassification' in r['body']
                                   or NEXIS_TAXONOMY_RE.search(r['body']))]

pct = lambda n: f'{100 * n / max(len(all_records), 1):.1f}%'

print(f'Total records           : {len(all_records)}')
print(f'Null date               : {len(null_date)}  ({pct(len(null_date))})')
print(f'Null source             : {len(null_source)}  ({pct(len(null_source))})')
print(f'Null headline           : {len(null_headline)}  ({pct(len(null_headline))})')
print(f'Bodies with leaked meta : {len(leaked_meta)}  (must be 0)')

if leaked_meta:
    print('\n--- Leaked-metadata samples (Step 3 regex needs adjustment) ---')
    for r in leaked_meta[:3]:
        i, label = r['body'].find('\nClassification'), 'Classification'
        if i < 0:
            m = NEXIS_TAXONOMY_RE.search(r['body'])
            i, label = m.start(), m.group(0).strip().split(':')[0]
        print(f"  id={r['id']}  {label} found at body offset {i}")
        print(f'  ...{r["body"][max(0,i-30):i+80]!r}...')

if null_date:
    print('\n--- Records with null date (inspect manually) ---')
    for r in null_date[:5]:
        print(f"  id={r['id']}  source={r['source']!r}  headline={r['headline']!r}")

print('\n--- First 3 parsed records ---')
for r in all_records[:3]:
    print(f"  id       : {r['id']}")
    print(f"  date     : {r['date']}")
    print(f"  source   : {r['source']}")
    print(f"  headline : {r['headline']}")
    print(f"  body[:120]: {(r['body'] or '')[:120]}")
    print(f"  body[-80:]: {(r['body'] or '')[-80:]}")
    print()

## Step 6: Write corpus.jsonl

In [ ]:
with open(OUT_FILE, 'w', encoding='utf-8') as f:
    for record in all_records:
        f.write(json.dumps(record, ensure_ascii=False) + '\n')

print(f'Wrote {len(all_records)} records to {OUT_FILE}')
print()
print('VALIDATION CHECKPOINT (01_parse):')
print(f'  Total articles          : {len(all_records)}')
print(f'  Null dates              : {len(null_date)} ')
print(f'  Bodies with leaked meta : {len(leaked_meta)} ')